In [1]:
pip uninstall snscrape -y

Found existing installation: snscrape 0.7.0.20230622
Uninstalling snscrape-0.7.0.20230622:
  Successfully uninstalled snscrape-0.7.0.20230622
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install git+https://github.com/JustAnotherArchivist/snscrape.git

  Cloning https://github.com/JustAnotherArchivist/snscrape.git to c:\users\pnboo\appdata\local\temp\pip-req-build-a00jl8mh
  Resolved https://github.com/JustAnotherArchivist/snscrape.git to commit 614d4c2029a62d348ca56598f87c425966aaec66
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for snscrape: filename=snscrape-0.7.0.20230622-py3-none-any.whl size=75387 sha256=5981b22227bd14b8a4dfcd5a08f730748b041e6bfeb8252d2b7db85542ed7d3d
  Stored in directory: C:\Users\pnboo\AppData\Local\Temp\pip-ephem-wheel-cache-s4v_khv7\wheels\22\b0\83\c1246b082245e5800613628731f005822e06e89e41f967527f
Successfully built snscrape
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/JustAnotherArchivist/snscrape.git 'C:\Users\pnboo\AppData\Local\Temp\pip-req-build-a00jl8mh'

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# ----------------------------
# Option B: Dynamic loader (non-destructive) to use snscrape modules
# ----------------------------
import sys, pathlib, importlib.util, types
import pkgutil, json

# locate installed snscrape package
import pkgutil
loader = pkgutil.find_loader("snscrape")
if loader is None:
    raise RuntimeError("Could not find snscrape package in this environment. Install it first.")
import snscrape
snscrape_path = pathlib.Path(snscrape.__file__).resolve().parent
modules_dir = snscrape_path / "modules"
print("snscrape modules dir:", modules_dir)

# load each .py file in modules into a module object and insert into sys.modules with name 'snscrape.modules.<modname>'
for p in sorted(modules_dir.glob("*.py")):
    if p.name == "__init__.py":
        continue
    modname = p.stem
    fqname = f"snscrape.modules.{modname}"
    if fqname in sys.modules:
        print(f"{fqname} already loaded; skipping.")
        continue
    spec = importlib.util.spec_from_file_location(fqname, str(p))
    mod = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(mod)
        sys.modules[fqname] = mod
        print("Loaded", fqname)
    except Exception as e:
        print("Failed to load", p, ":", e)

# Now try importing the twitter module (it will pick up our loaded module object)
try:
    import snscrape.modules.twitter as sntwitter
    print("Imported snscrape.modules.twitter OK — file:", getattr(sntwitter, '__file__', 'unknown'))
except Exception as e:
    print("Import failed:", e)


C:\Users\pnboo\AppData\Local\Temp\ipykernel_13336\1536739239.py:9: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  loader = pkgutil.find_loader("snscrape")


snscrape modules dir: D:\Documents\University\Senior Project\gender-bias-detection\.venv\Lib\site-packages\snscrape\modules
Loaded snscrape.modules.facebook
Loaded snscrape.modules.instagram
Failed to load D:\Documents\University\Senior Project\gender-bias-detection\.venv\Lib\site-packages\snscrape\modules\mastodon.py : 'NoneType' object has no attribute '__dict__'
Loaded snscrape.modules.reddit
Loaded snscrape.modules.telegram
Failed to load D:\Documents\University\Senior Project\gender-bias-detection\.venv\Lib\site-packages\snscrape\modules\twitter.py : 'NoneType' object has no attribute '__dict__'
Loaded snscrape.modules.vkontakte
Loaded snscrape.modules.weibo
Import failed: 'FileFinder' object has no attribute 'find_module'


In [ ]:
# Notebook-safe snscrape (CLI) scraper that avoids `import snscrape`
import sys, subprocess, json, shlex, re, html, time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional, ClassVar, List
import pandas as pd
from pprint import pprint

# -------------------------
# CONFIG
# -------------------------
TWEET_URL = "https://x.com/Ravenmelonia/status/1975951650046312675"  # <-- replace with your tweet URL
MAX_REPLIES = 500
MAX_QUOTES = 500
SNSCRAPE_GIT = "git+https://github.com/JustAnotherArchivist/snscrape.git"

# -------------------------
# Dataclass (standalone; uses your schema)
# -------------------------
FIELDS = [
    "id","platform","platform_type","url","content_type",
    "timestamp","scraper_module","text","raw_text","scrape_date"
]
SCRAPE_DATE = datetime.now(timezone.utc).isoformat()

@dataclass
class SocialMediaRecord:
    id: Optional[str] = None
    platform: Optional[str] = "twitter"
    platform_type: Optional[str] = None
    url: Optional[str] = None
    content_type: Optional[str] = "text"
    timestamp: Optional[str] = None
    scraper_module: Optional[str] = "snscrape-cli"
    text: Optional[str] = None
    raw_text: Optional[str] = None
    scrape_date: str = field(default_factory=lambda: SCRAPE_DATE)

    FIELDS: ClassVar[list[str]] = FIELDS
    def to_dict(self): return asdict(self)
    @classmethod
    def get_fields(cls): return cls.FIELDS

# -------------------------
# Helpers
# -------------------------
def tweet_id_from_url(url: str):
    """Return (username, id) where available"""
    m = re.search(r"https?://(?:www\.)?(?:x\.com|twitter\.com)/([^/]+)/status/(\d+)", url)
    if not m:
        # try more permissive match for id
        m2 = re.search(r"/status/(\d+)", url)
        if not m2:
            return (None, None)
        return (None, m2.group(1))
    return (m.group(1), m.group(2))

def clean_text_for_nlp(text: str) -> str:
    """Keep emojis; remove URLs/mentions/extra whitespace"""
    if not text: return ""
    s = html.unescape(text)
    s = re.sub(r"http\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def run_cmd(cmd: List[str], input_text: str = None, timeout: int = 300):
    """Run subprocess command, return (returncode, stdout, stderr)"""
    try:
        proc = subprocess.run(cmd, input=input_text, capture_output=True, text=True, timeout=timeout)
        return proc.returncode, proc.stdout, proc.stderr
    except subprocess.TimeoutExpired as e:
        return 1, "", f"timeout: {e}"

def ensure_snscrape_cli_installed(try_install: bool = True):
    """Check if `python -m snscrape --version` works; if not optionally pip install from GitHub."""
    cmd = [sys.executable, "-m", "snscrape", "--version"]
    rc, out, err = run_cmd(cmd)
    if rc == 0:
        return True
    if not try_install:
        return False
    print("snscrape CLI not available; attempting to install from GitHub (may take a minute)...")
    rc, out, err = run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", SNSCRAPE_GIT])
    if rc != 0:
        print("pip install failed:", err)
        return False
    # small pause to let console script be discoverable
    time.sleep(1.0)
    rc2, out2, err2 = run_cmd([sys.executable, "-m", "snscrape", "--version"])
    if rc2 == 0:
        print("snscrape CLI installed.")
        return True
    print("snscrape still not available after install. stderr:", err2)
    return False

def run_snscrape_query_json(query: str, max_results: int = 1000):
    """Run snscrape and return list of parsed JSON objects (each line is a JSON object)."""
    cmd_str = f"{sys.executable} -m snscrape --jsonl --max-results {max_results} twitter-search {shlex_quote(query)}"
    # Use shell for convenience (keeps quoting simple)
    proc = subprocess.Popen(cmd_str, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    results = []
    # read stdout line-by-line to avoid huge memory use
    for line in proc.stdout:
        line = line.strip()
        if not line:
            continue
        try:
            results.append(json.loads(line))
        except Exception:
            # ignore parse errors but show a short preview for debugging
            pass
    proc.stdout.close()
    proc.wait()
    if proc.returncode != 0:
        stderr = proc.stderr.read()
        proc.stderr.close()
        # return empty but raise info
        # print("snscrape error (stderr):", stderr[:400])
        return [], stderr
    proc.stderr.close()
    return results, None

def shlex_quote(s: str):
    # simple helper to quote for shell invocation
    return "'" + s.replace("'", "'\"'\"'") + "'"

# -------------------------
# Scrape routines (use CLI)
# -------------------------
def fetch_original_by_fallbacks(tweet_url, tweet_id, username=None):
    """Try several CLI queries to fetch the original tweet JSON."""
    queries_tried = []
    # primary: id:<id>
    queries_tried.append(f"id:{tweet_id}")
    # fallback: from:username id:id (if username present)
    if username:
        queries_tried.append(f"from:{username} id:{tweet_id}")
    # fallback: full URL string as quoted phrase
    queries_tried.append(f'"{tweet_url}"')
    # fallback: search for id as plain number (less reliable)
    queries_tried.append(f"{tweet_id}")
    for q in queries_tried:
        objs, err = run_snscrape_query_json(q, max_results=2)
        if err:
            # continue to next if snscrape returned non-zero but still possibly produced output
            pass
        if objs:
            # choose the best match (prefer tweets where id matches)
            for o in objs:
                if str(o.get("id")) == str(tweet_id) or tweet_id in (o.get("url") or ""):
                    return o
            # otherwise return first object if nothing strictly matched
            return objs[0]
    return None

def scrape_tweet_and_context_cli(tweet_url: str, max_replies: int = 500, max_quotes: int = 500):
    # ensure CLI available
    if not ensure_snscrape_cli_installed(try_install=True):
        raise RuntimeError("snscrape CLI not available and installation failed. Please install `snscrape` manually in this environment.")
    username, tweet_id = tweet_id_from_url(tweet_url)
    if not tweet_id:
        raise ValueError("Could not parse tweet id from URL: " + tweet_url)

    records: List[SocialMediaRecord] = []

    # --- original tweet (fallback attempts) ---
    orig_json = fetch_original_by_fallbacks(tweet_url, tweet_id, username=username)
    if orig_json:
        rec = SocialMediaRecord(
            id=str(orig_json.get("id")),
            platform="twitter",
            platform_type="post",
            url=orig_json.get("url"),
            content_type=("text" if not orig_json.get("media") else "image" if any(m.get("type","").lower() in ("photo","image") for m in orig_json.get("media",[])) else "video"),
            timestamp=orig_json.get("date"),
            scraper_module="snscrape-cli",
            text=clean_text_for_nlp(orig_json.get("content")),
            raw_text=orig_json.get("content"),
        )
        records.append(rec)
    else:
        print("Original tweet not found via snscrape queries. You may have an invalid URL or the tweet is deleted/protected.")

    # --- replies (conversation) ---
    replies_objs, errr = run_snscrape_query_json(f"conversation_id:{tweet_id} -filter:retweets", max_results=max_replies)
    if errr:
        # keep going even if stderr non-empty
        pass
    for o in replies_objs:
        if str(o.get("id")) == str(tweet_id):
            continue
        rec = SocialMediaRecord(
            id=str(o.get("id")),
            platform="twitter",
            platform_type="reply",
            url=o.get("url"),
            content_type=("text" if not o.get("media") else "image" if any(m.get("type","").lower() in ("photo","image") for m in o.get("media",[])) else "video"),
            timestamp=o.get("date"),
            scraper_module="snscrape-cli",
            text=clean_text_for_nlp(o.get("content")),
            raw_text=o.get("content"),
        )
        records.append(rec)

    # --- quotes ---
    quotes_objs, errq = run_snscrape_query_json(f"quoted_tweet_id:{tweet_id} -filter:retweets", max_results=max_quotes)
    if errq:
        pass
    for o in quotes_objs:
        rec = SocialMediaRecord(
            id=str(o.get("id")),
            platform="twitter",
            platform_type="quote",
            url=o.get("url"),
            content_type=("text" if not o.get("media") else "image" if any(m.get("type","").lower() in ("photo","image") for m in o.get("media",[])) else "video"),
            timestamp=o.get("date"),
            scraper_module="snscrape-cli",
            text=clean_text_for_nlp(o.get("content")),
            raw_text=o.get("content"),
        )
        records.append(rec)

    # Convert to DataFrame
    rows = [r.to_dict() for r in records]
    if not rows:
        print("No rows scraped. Possible reasons: invalid tweet URL, tweet deleted/protected, or snscrape couldn't find the tweet.")
    df = pd.DataFrame(rows)
    for c in SocialMediaRecord.get_fields():
        if c not in df.columns:
            df[c] = None
    df = df[SocialMediaRecord.get_fields()]
    return df

# -------------------------
# Run the scraper
# -------------------------
df = scrape_tweet_and_context_cli(TWEET_URL, max_replies=MAX_REPLIES, max_quotes=MAX_QUOTES)
out = "tweet_and_context_cli.csv"
df.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to {out}")
pprint(df.head(10).to_dict(orient="records"))


snscrape CLI not available; attempting to install from GitHub (may take a minute)...
snscrape still not available after install. stderr: d:\Documents\University\Senior Project\gender-bias-detection\.venv\Scripts\python.exe: No module named snscrape.__main__; 'snscrape' is a package and cannot be directly executed



RuntimeError: snscrape CLI not available and installation failed. Please install `snscrape` manually in this environment.

In [ ]:
# ----------------------
# Imports
# ----------------------
import snscrape.modules.twitter as sntwitter
import pandas as pd
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional, ClassVar
import html
import re
from pprint import pprint

# ----------------------
# Constants
# ----------------------
FIELDS = [
    "id",
    "platform",
    "platform_type",
    "url",
    "content_type",
    "timestamp",
    "scraper_module",
    "text",
    "raw_text",
    "scrape_date"
]
SCRAPE_DATE = datetime.now(timezone.utc).isoformat()
DATETIME_FORMAT = "%Y-%m-%dT%H:%M:%S%z"

# ----------------------
# Dataclass
# ----------------------
@dataclass
class SocialMediaRecord:
    id: Optional[str] = None
    platform: Optional[str] = None
    platform_type: Optional[str] = None
    url: Optional[str] = None
    content_type: Optional[str] = "text"
    timestamp: Optional[str] = None
    scraper_module: Optional[str] = None
    text: Optional[str] = None
    raw_text: Optional[str] = None
    scrape_date: str = field(default_factory=lambda: SCRAPE_DATE)

    FIELDS: ClassVar[list[str]] = FIELDS

    def to_dict(self):
        return asdict(self)

    @classmethod
    def get_fields(cls):
        return cls.FIELDS

# ----------------------
# Helpers
# ----------------------
def tweet_id_from_url(url: str) -> str:
    m = re.search(r"/status/(\d+)", url)
    if not m:
        raise ValueError(f"Cannot parse tweet ID from URL: {url}")
    return m.group(1)

def clean_text_for_nlp(text: str) -> str:
    if not text:
        return ""
    text = html.unescape(text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def detect_content_type(tweet) -> str:
    try:
        if hasattr(tweet, "media") and tweet.media:
            for m in tweet.media:
                if getattr(m, "mediaType", "").lower() in ("photo", "image"):
                    return "image"
                if getattr(m, "mediaType", "").lower() in ("video", "animated_gif"):
                    return "video"
            return "image"
    except:
        pass
    return "text"

# ----------------------
# Scraping functions
# ----------------------
def scrape_original(tweet_id: str):
    results = []
    for t in sntwitter.TwitterTweetScraper(tweet_id).get_items():
        rec = SocialMediaRecord(
            id=str(t.id),
            platform="twitter",
            platform_type="post",
            url=t.url,
            content_type=detect_content_type(t),
            timestamp=t.date.isoformat() if t.date else None,
            scraper_module="snscrape",
            text=clean_text_for_nlp(t.content),
            raw_text=t.content,
        )
        results.append(rec)
        break  # only one original tweet
    return results

def scrape_replies(tweet_id: str, max_results=500):
    records = []
    query = f"conversation_id:{tweet_id} -filter:retweets"
    for i, t in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
        if i >= max_results:
            break
        if str(t.id) == str(tweet_id):
            continue  # skip root
        rec = SocialMediaRecord(
            id=str(t.id),
            platform="twitter",
            platform_type="reply",
            url=t.url,
            content_type=detect_content_type(t),
            timestamp=t.date.isoformat() if t.date else None,
            scraper_module="snscrape",
            text=clean_text_for_nlp(t.content),
            raw_text=t.content,
        )
        records.append(rec)
    return records

def scrape_quotes(tweet_id: str, max_results=500):
    records = []
    query = f"quoted_tweet_id:{tweet_id} -filter:retweets"
    for i, t in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
        if i >= max_results:
            break
        rec = SocialMediaRecord(
            id=str(t.id),
            platform="twitter",
            platform_type="quote",
            url=t.url,
            content_type=detect_content_type(t),
            timestamp=t.date.isoformat() if t.date else None,
            scraper_module="snscrape",
            text=clean_text_for_nlp(t.content),
            raw_text=t.content,
        )
        records.append(rec)
    return records

# ----------------------
# High-level runner
# ----------------------
def scrape_tweet_and_context(tweet_url: str, max_replies=500, max_quotes=500):
    tweet_id = tweet_id_from_url(tweet_url)
    all_records = []

    all_records.extend(scrape_original(tweet_id))
    all_records.extend(scrape_replies(tweet_id, max_results=max_replies))
    all_records.extend(scrape_quotes(tweet_id, max_results=max_quotes))

    rows = [r.to_dict() for r in all_records]
    df = pd.DataFrame(rows)
    for c in SocialMediaRecord.get_fields():
        if c not in df.columns:
            df[c] = None
    df = df[SocialMediaRecord.get_fields()]
    return df

# ----------------------
# Example usage
# ----------------------
TWEET_URL = "https://x.com/Ravenmelonia/status/1975951650046312675"  # replace with real tweet
df = scrape_tweet_and_context(TWEET_URL, max_replies=200, max_quotes=200)
df.to_csv("tweet_and_context_api.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to CSV")
pprint(df.head(10).to_dict(orient="records"))


Saved 0 rows to CSV
[]


vvvv

In [ ]:
# CELL 2: Robust scraper (error recovery + HTML snapshot) and conversion to SocialMediaRecord
#         — RAW text (no cleaning, no separators injected)

import time, re
from urllib.parse import urljoin
from bs4 import BeautifulSoup

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException

import pandas as pd

# === 1) IMPORT YOUR DATACLASS (adjust this import to your project structure) ===
# Example if your file is src/dataset/models/social_media_record.py:
# from src.dataset.models.social_media_record import SocialMediaRecord
from scraper.dataclass import SocialMediaRecord  # <-- change to your actual path if needed

# --- constants for mapping ---
SCRAPER_NAME = "selenium_x_scraper_v1"
PLATFORM = "twitter"         # keep consistent with your dataset
DEFAULT_PLATFORM_TYPE = "post"
CONTENT_TYPE = "text"

# ---------- Helpers ----------
def _num_from_label(s: str | None):
    if not s: return None
    s = s.strip().replace(",", "")
    m = re.search(r'([\d]+(?:\.\d+)?)\s*([KkMm])?', s)
    if not m: return None
    val = float(m.group(1))
    suf = (m.group(2) or "").lower()
    if suf == "k": val *= 1_000
    elif suf == "m": val *= 1_000_000
    return int(val)

def _is_error_screen(drv) -> bool:
    try:
        err = drv.find_elements(By.XPATH, "//*[contains(., 'Something went wrong')]")
        return bool(err)
    except Exception:
        return False

def _ensure_logged_in(drv, timeout=12) -> bool:
    try:
        WebDriverWait(drv, timeout).until(
            EC.any_of(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="SideNav_NewTweet_Button"], [data-testid="tweetButtonInline"]')),
                EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href^="/compose/tweet"]')),
                EC.presence_of_element_located((By.CSS_SELECTOR, 'a[aria-label*="Profile"]'))
            )
        )
        return True
    except TimeoutException:
        return False

def _wait_for_tweets(drv, timeout=12) -> bool:
    try:
        WebDriverWait(drv, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'article[role="article"]'))
        )
        return True
    except TimeoutException:
        return False

def _safe_reload_until_ok(drv, target_url, tries=4, settle=1.2) -> bool:
    for i in range(tries):
        if i == 0:
            drv.get(target_url)
        else:
            drv.refresh()
        time.sleep(1.0)
        if _is_error_screen(drv):
            time.sleep(1.0)
            continue
        if _wait_for_tweets(drv, timeout=10):
            time.sleep(settle)
            return True
    return False

# ---------- Parsing via HTML snapshot (RAW text) ----------
def parse_article_via_html(driver, el):
    html = driver.execute_script("return arguments[0].outerHTML;", el)
    soup = BeautifulSoup(html, "html.parser")

    row = {
        "tweet_id": None, "timestamp_iso": None,
        "user_display_name": "", "user_handle": "",
        "text": "", "like_count": None, "retweet_count": None,
        "reply_count": None, "view_count": None,
        "permalink": "", "key": None
    }

    # name/handle
    name_block = soup.select_one('div[data-testid="User-Name"], div[data-testid="User-Names"]')
    if name_block:
        spans = name_block.select("span")
        for sp in spans:
            txt = sp.get_text()
            if not txt:
                continue
            if txt.startswith("@"):
                row["user_handle"] = txt
            elif not row["user_display_name"]:
                row["user_display_name"] = txt

    # text — RAW (no cleaning, no added separators)
    tw = soup.select_one('div[data-testid="tweetText"]')
    if tw:
        # NOTE: we do NOT modify <br> or inject separators; this is as raw as get_text allows.
        raw = tw.get_text(separator="", strip=False)
        row["text"] = raw

    # time
    t = soup.select_one("time")
    if t and t.has_attr("datetime"):
        row["timestamp_iso"] = t["datetime"]

    # link & id
    a = soup.select_one('a[href*="/status/"]')
    if a and a.has_attr("href"):
        href = a["href"]
        if href.startswith("/"):
            href = urljoin("https://x.com", href)
        row["permalink"] = href
        m = re.search(r"/status/(\d+)", href)
        if m:
            row["tweet_id"] = m.group(1)

    # counts
    def c(selector):
        btn = soup.select_one(selector)
        if not btn: return None
        lab = btn.get("aria-label") or btn.get_text()
        return _num_from_label(lab)
    row["reply_count"]   = c('[data-testid="reply"]')
    row["retweet_count"] = c('[data-testid="retweet"]')
    row["like_count"]    = c('[data-testid="like"]')

    # views
    for s in soup.select('span[aria-label]'):
        lab = s.get("aria-label") or ""
        if "View" in lab:
            vc = _num_from_label(lab)
            if vc is not None:
                row["view_count"] = vc
                break

    row["key"] = row["tweet_id"] or f"{row['user_handle']}|{row['timestamp_iso']}|{row['text'][:50]}"
    return row

# ---------- Scraper ----------
def scrape_tweets(driver, target_url, max_scrolls=60, pause_secs=1.2, cooldown_every=15, cooldown_secs=3.0):
    if not _ensure_logged_in(driver, timeout=15):
        print("It doesn’t look like you’re logged in. Please log in in the Chrome window first.")
        return []

    if not _safe_reload_until_ok(driver, target_url, tries=5, settle=1.4):
        print("Couldn’t get past the error screen for that URL.")
        return []

    results, seen = [], set()
    not_growing, last_seen = 0, 0

    def get_articles(drv):
        return drv.find_elements(By.CSS_SELECTOR, 'article[role="article"]')

    for s in range(max_scrolls):
        if _is_error_screen(driver):
            if not _safe_reload_until_ok(driver, driver.current_url, tries=3, settle=1.2):
                print("Hit persistent error page mid-run; stopping.")
                break

        articles = get_articles(driver)
        for idx in range(len(articles)):
            try:
                art = get_articles(driver)[idx]
                row = parse_article_via_html(driver, art)
            except StaleElementReferenceException:
                time.sleep(0.1)
                continue
            except Exception:
                continue

            if not (row["tweet_id"] or row["text"]):
                continue
            if row["key"] in seen:
                continue

            seen.add(row["key"])
            results.append(row)
            print(
                f"[{row['timestamp_iso']}] {row['user_display_name']} {row['user_handle']} "
                f"♥{row['like_count']} ↻{row['retweet_count']} 💬{row['reply_count']} 👁️{row['view_count']}\n"
                f"{row['text']}\n{row['permalink']}\n" + "—"*40
            )

        if len(seen) == last_seen:
            not_growing += 1
        else:
            not_growing = 0
        last_seen = len(seen)
        if not_growing >= 6:
            break

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause_secs)

        if cooldown_every and (s + 1) % cooldown_every == 0:
            time.sleep(cooldown_secs)

    print(f"\nCollected {len(results)} unique tweets.")
    return results

# ---------- Mapping to SocialMediaRecord (NO CSV WRITE; RAW text) ----------
def infer_platform_type(tweet_text: str) -> str:
    if (tweet_text or "").strip().startswith("RT @"):
        return "retweet"
    return DEFAULT_PLATFORM_TYPE  # "post" by default

def tweet_to_record(t: dict) -> SocialMediaRecord:
    raw = t.get("text") or ""
    return SocialMediaRecord(
        id=t.get("tweet_id") or t.get("key"),
        platform=PLATFORM,
        platform_type=infer_platform_type(raw),
        url=t.get("permalink") or "",
        content_type=CONTENT_TYPE,
        timestamp=t.get("timestamp_iso") or "",
        scraper_module=SCRAPER_NAME,
        text=raw,        # RAW
        raw_text=raw,    # RAW
        # scrape_date uses your SCRAPE_DATE default
    )

def tweets_to_dataframe_social(tweets: list[dict]) -> pd.DataFrame:
    records = [tweet_to_record(t) for t in tweets]
    df = pd.DataFrame([r.to_dict() for r in records])
    try:
        df = df[SocialMediaRecord.get_fields()]  # canonical header order, if provided by your class
    except Exception:
        pass
    return df

# === Use it ===
TARGET_URL = "https://x.com/search?q=ข้าวมันไก่&f=live"
raw_items = scrape_tweets(driver, TARGET_URL, max_scrolls=80, pause_secs=1.2)

records_df = tweets_to_dataframe_social(raw_items)


vvvvvvvvv

In [ ]:


# Replace with your CSV file path
csv_file = "D:\Documents\University\Senior Project\gender-bias-detection\services\scraper\src\scraper\blog_based\tweets_data.csv"

# Load the CSV into a DataFrame
df = pd.read_csv(csv_file)

# Show the first few rows (optional)
display(df.head())

# Count number of rows
num_rows = len(df)
print(f"The CSV has {num_rows} rows.")
